# ReactOT-Style Baseline (No Masking)

This notebook mirrors the no-masking flow-matching setup from `deepprinciple-react-ot-5184332/reactOT.ipynb`.
It uses a fixed atom count (most common = 10), and a simple conditional Flow-EGNN with training steps matching the original.

In [1]:
import pickle
import math
import numpy as np
import torch
import torch.nn as nn

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)


Using device: mps


In [2]:
pkl_path = "../Data/train_rpsb_all.pkl"
with open(pkl_path, "rb") as f:
    obj = pickle.load(f)

print(obj.keys())
print(obj["reactant"].keys())


dict_keys(['reactant', 'transition_state', 'product', 'single_fragment', 'use_ind', 'ts_guess', 'ts_guess_sbv1', 'ts_guess_true', 'ts_guess_NEBCI-xtb'])
dict_keys(['num_atoms', 'charges', 'fragments', 'positions', 'rxn', 'wB97x_6-31G(d).energy', 'wB97x_6-31G(d).atomization_energy', 'wB97x_6-31G(d).forces', 'formula', 'xtb_positions'])


In [3]:
# no masking: select most common atom count
Ns = [obj["reactant"]["positions"][i].shape[0] for i in range(2000)]
from collections import Counter
cnt = Counter(Ns)
common_N, common_count = cnt.most_common(1)[0]
print("Most common N:", common_N, "count in first 2000:", common_count)

idxs = [i for i in range(len(obj["reactant"]["positions"]))
        if obj["reactant"]["positions"][i].shape[0] == common_N]
print("Total reactions with that N:", len(idxs))
print("Example indices:", idxs[:10])


Most common N: 10 count in first 2000: 560
Total reactions with that N: 674
Example indices: [122, 123, 124, 125, 126, 127, 128, 129, 130, 131]


In [4]:
def get_batch(indices, batch_size=16):
    sel = indices[:batch_size]
    xR = np.stack([obj["reactant"]["positions"][i] for i in sel], axis=0)
    xP = np.stack([obj["product"]["positions"][i] for i in sel], axis=0)
    xT = np.stack([obj["transition_state"]["positions"][i] for i in sel], axis=0)
    z  = np.stack([np.array(obj["reactant"]["charges"][i], dtype=np.int64) for i in sel], axis=0)

    xR = torch.tensor(xR, dtype=torch.float32, device=device)
    xP = torch.tensor(xP, dtype=torch.float32, device=device)
    xT = torch.tensor(xT, dtype=torch.float32, device=device)
    z  = torch.tensor(z,  dtype=torch.long, device=device)
    return xR, xP, xT, z

xR, xP, xT, z = get_batch(idxs, batch_size=8)
print(xR.shape, xP.shape, xT.shape, z.shape, xR.dtype, z.dtype, xR.device)


torch.Size([8, 10, 3]) torch.Size([8, 10, 3]) torch.Size([8, 10, 3]) torch.Size([8, 10]) torch.float32 torch.int64 mps:0


In [5]:
def make_flow_batch(xR, xP, xT, z):
    x0 = 0.5 * (xR + xP)
    x1 = xT
    B  = xR.shape[0]
    t  = torch.rand(B, device=xR.device)
    x_t = (1 - t)[:, None, None] * x0 + t[:, None, None] * x1
    v_star = x1 - x0
    cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP}
    return x_t, t, cond, v_star

x_t, t, cond, v_star = make_flow_batch(xR, xP, xT, z)
print("x_t", x_t.shape, "t", t.shape, "v*", v_star.shape)
print("t min/max:", float(t.min()), float(t.max()))


x_t torch.Size([8, 10, 3]) t torch.Size([8]) v* torch.Size([8, 10, 3])
t min/max: 0.17483627796173096 0.9171103239059448


In [6]:
class TimeEmbedding(nn.Module):
    def __init__(self, n_freq=16):
        super().__init__()
        self.register_buffer("freqs", 2 * math.pi * (2 ** torch.arange(n_freq).float()))
    def forward(self, t):
        args = t[:, None] * self.freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

te = TimeEmbedding(8).to(device)
out = te(t)
print(out.shape)
print(out[0, :10])


torch.Size([8, 16])
tensor([ 0.9384,  0.6485, -0.9873, -0.3137,  0.5958,  0.9570,  0.5552, -0.9235,
         0.3455, -0.7612], device='mps:0')


In [7]:
class EGNNLayer(nn.Module):
    def __init__(self, h_dim):
        super().__init__()
        self.phi_m = nn.Sequential(
            nn.Linear(2*h_dim + 1, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
        )
        self.phi_h = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
        )
        self.phi_x = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 1),
        )

    def forward(self, h, x, src, dst):
        B, N, H = h.shape
        xi = x[:, src, :]
        xj = x[:, dst, :]
        rij = xi - xj
        dij = (rij ** 2).sum(dim=-1, keepdim=True)

        hi = h[:, src, :]
        hj = h[:, dst, :]
        mij = self.phi_m(torch.cat([hi, hj, dij], dim=-1))

        agg = torch.zeros((B, N, H), device=h.device, dtype=h.dtype)
        agg.scatter_add_(1, src[None, :, None].expand(B, -1, H), mij)

        h = h + self.phi_h(agg)

        s = self.phi_x(mij)
        dx_ij = rij * s
        dx = torch.zeros((B, N, 3), device=x.device, dtype=x.dtype)
        dx.scatter_add_(1, src[None, :, None].expand(B, -1, 3), dx_ij)
        x = x + dx
        return h, x

def fully_connected_edges(N, device):
    idx = torch.arange(N, device=device)
    ii, jj = torch.meshgrid(idx, idx, indexing="ij")
    mask = ii != jj
    src = ii[mask].reshape(-1)
    dst = jj[mask].reshape(-1)
    return src, dst


In [8]:
class FlowEGNN(nn.Module):
    def __init__(self, z_max=100, h_dim=128, n_layers=4, time_freq=16, use_concat_ctx=True):
        super().__init__()
        self.use_concat_ctx = use_concat_ctx
        self.z_emb = nn.Embedding(z_max+1, h_dim)

        self.t_emb = TimeEmbedding(n_freq=time_freq)
        self.t_proj = nn.Linear(2*time_freq, h_dim)

        if use_concat_ctx:
            self.ctx_proj = nn.Linear(9, h_dim)
        else:
            self.ctx_proj = nn.Linear(3, h_dim)

        self.layers = nn.ModuleList([EGNNLayer(h_dim) for _ in range(n_layers)])

        self.v_head = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 3)
        )

    def forward(self, x_t, t, cond):
        z = cond["z"]
        xR = cond["xR_ctx"]
        xP = cond["xP_ctx"]

        B, N, _ = x_t.shape
        h = self.z_emb(z)

        te = self.t_proj(self.t_emb(t))
        h = h + te[:, None, :].expand(B, N, -1)

        diff = xP - xR
        if self.use_concat_ctx:
            ctx = torch.cat([xR, xP, diff], dim=-1)
        else:
            ctx = diff
        h = h + self.ctx_proj(ctx)

        src, dst = fully_connected_edges(N, x_t.device)
        x = x_t
        for layer in self.layers:
            h, x = layer(h, x, src, dst)

        v = self.v_head(h)
        return v


In [9]:
model_diff = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=False).to(device)
model_cat  = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=True ).to(device)

v1 = model_diff(x_t, t, cond)
v2 = model_cat(x_t, t, cond)

print(v1.shape, v2.shape)
print("v1 norm mean:", float(v1.norm(dim=-1).mean()))
print("v2 norm mean:", float(v2.norm(dim=-1).mean()))


torch.Size([8, 10, 3]) torch.Size([8, 10, 3])
v1 norm mean: 0.4055713713169098
v2 norm mean: 0.3963898718357086


/var/folders/8p/6t_fjn5d1c7fxwflz29kdvqr0000gn/T/ipykernel_32006/3374611194.py:8: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print("v1 norm mean:", float(v1.norm(dim=-1).mean()))


In [10]:
def flow_loss(v_pred, v_star):
    return ((v_pred - v_star) ** 2).mean()

opt = torch.optim.Adam(model_cat.parameters(), lr=1e-3)

opt.zero_grad()
v_pred = model_cat(x_t, t, cond)
loss = flow_loss(v_pred, v_star)
loss.backward()

grad_norms = []
for name, p in model_cat.named_parameters():
    if p.grad is not None:
        grad_norms.append((name, float(p.grad.norm().detach().cpu())))
print("loss:", float(loss.detach().cpu()))
print("some grad norms:", grad_norms[:5])


loss: 0.16508516669273376
some grad norms: [('z_emb.weight', 0.027388721704483032), ('t_proj.weight', 0.06450781971216202), ('t_proj.bias', 0.03532533720135689), ('ctx_proj.weight', 0.055339064449071884), ('ctx_proj.bias', 0.03532533720135689)]


In [11]:
model = model_cat
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

losses = []
for step in range(200):
    xR, xP, xT, z = get_batch(idxs, batch_size=32)
    x_t, t, cond, v_star = make_flow_batch(xR, xP, xT, z)

    opt.zero_grad()
    v_pred = model(x_t, t, cond)
    loss = ((v_pred - v_star)**2).mean()
    loss.backward()
    opt.step()

    losses.append(float(loss.detach().cpu()))
    if step % 20 == 0:
        print(step, losses[-1])


0 0.13794228434562683
20 0.07306269556283951
40 0.06755601614713669
60 0.06273206323385239
80 0.05780365690588951
100 0.05155413597822189
120 0.043423671275377274
140 0.033506203442811966
160 0.026930542662739754
180 0.021932801231741905


In [13]:
@torch.no_grad()
def euler_solve(model, x0, cond, steps=50):
    x = x0
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((x.shape[0],), k*dt, device=x.device)
        v = model(x, t, cond)
        x = x + dt * v
    return x

xR, xP, xT, z = get_batch(idxs, batch_size=16)
x0 = 0.5 * (xR + xP)
cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP}

xTS_pred = euler_solve(model, x0, cond, steps=80)

def rmsd_torch(A, B):
    return torch.sqrt(((A - B)**2).sum(dim=-1).mean(dim=-1))

r_mid = rmsd_torch(x0, xT).mean()
r_pred = rmsd_torch(xTS_pred, xT).mean()
print("RMSD midpoint->TS:", float(r_mid))
print("RMSD pred->TS    :", float(r_pred))


RMSD midpoint->TS: 0.42418259382247925
RMSD pred->TS    : 0.21741563081741333
